In [ ]:
# ==========================================
# 1. 필수 라이브러리 설치
# ==========================================
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


In [ ]:
import torch
print("GPU 사용 가능:", torch.cuda.is_available())
print("GPU 이름:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

GPU 사용 가능: True
GPU 이름: Tesla T4


In [ ]:
# ==========================================
# 2. 구글 드라이브 연동
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    PreTrainedTokenizerFast,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

Mounted at /content/drive


In [ ]:
# ==========================================
# 3. 데이터 로드
# ==========================================
data_path = '/content/drive/MyDrive/시냅스 팀플_2/dataset/train.csv'
df = pd.read_csv(data_path)

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset   = Dataset.from_pandas(val_df,   preserve_index=False)

print(f"학습 데이터: {len(train_dataset)}개 | 검증 데이터: {len(val_dataset)}개")

학습 데이터: 10136개 | 검증 데이터: 1127개


In [ ]:
# ==========================================
# 4. KoBART 모델 & 토크나이저 로드
# ==========================================
MODEL_NAME = "gogamza/kobart-base-v2"

tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_NAME)
model     = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

print("KoBART 로드 완료!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/682k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/495M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

KoBART 로드 완료!


In [ ]:
# ==========================================
# 5. 토크나이징
# ==========================================
MAX_LENGTH = 128  # 256은 너무 길어서 느림 — 128로 충분

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False,        # ← padding은 DataCollator가 담당
    )
    labels = tokenizer(
        text_target=examples["output"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_function, batched=True,
    remove_columns=train_dataset.column_names
)
tokenized_val = val_dataset.map(
    preprocess_function, batched=True,
    remove_columns=val_dataset.column_names
)

print("토크나이징 완료!")

Map:   0%|          | 0/10136 [00:00<?, ? examples/s]

Map:   0%|          | 0/1127 [00:00<?, ? examples/s]

토크나이징 완료!


In [ ]:
# ==========================================
# 6. F1 계산 (Counter 버전 — 버그 수정)
# ==========================================
def calc_char_f1(pred_str, true_str):
    pred_chars = list(pred_str.replace(" ", ""))
    true_chars = list(true_str.replace(" ", ""))

    if len(pred_chars) == 0 and len(true_chars) == 0:
        return 1.0
    if len(pred_chars) == 0 or len(true_chars) == 0:
        return 0.0

    pred_counter = Counter(pred_chars)
    true_counter = Counter(true_chars)
    common = sum((pred_counter & true_counter).values())  # 중복 포함 교집합

    precision = common / len(pred_chars)
    recall    = common / len(true_chars)

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    f1_scores = [calc_char_f1(p, l) for p, l in zip(decoded_preds, decoded_labels)]
    return {"char_f1": np.mean(f1_scores)}

In [ ]:
# ==========================================
# 7. 학습 세팅
# ==========================================
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_kobart",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    warmup_steps=100,
    save_total_limit=2,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=True,
    logging_steps=50,
    report_to="none",
    generation_max_length=128,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# ==========================================
# 8. 학습 시작
# ==========================================
print("KoBART 학습 시작!")
trainer.train()

KoBART 학습 시작!


Epoch,Training Loss,Validation Loss,Char F1
1,0.569385,0.885300,0.433314


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]